In [1]:
import torch
import os
import gc

gc.collect()
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

from harreman_funcs import HarremanRunner
import pandas as pd

In [2]:
XENIUM_DATA_DIR = '/global/scratch/users/fosterangus/MetabTravLR/Data/Xenium'
# DATASET_NAME = 'Primary_Dermal_Melanoma'
DATASET_NAME = 'Human_Lung'
DATA_DIR = f'{XENIUM_DATA_DIR}/{DATASET_NAME}'

In [3]:
harRunner = HarremanRunner(DATA_DIR)

In [4]:
harRunner.load_adata()

In [5]:
harRunner.save_harreman_network()

In [6]:
harRunner.run_harreman('Tier3')

running cell independent
Extracting interaction database...
Finished extracting interaction database in 0.371 seconds
Applying gene filtering...
Finished applying gene filtering in 0.000 seconds
Computing the neighborhood graph...
Computing the weights...
Finished computing the KNN graph in 2.744 seconds
Computing gene pairs...
Finished computing gene pairs in 0.120 seconds
Gene pairs to test: 416
Starting cell-cell communication analysis...
Running the parametric test...
Parametric test finished.
Running the non-parametric test...


Permutation test:   0%|          | 0/1000 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 884.00 MiB. GPU 0 has a total capacity of 10.57 GiB of which 395.06 MiB is free. Process 3007776 has 180.00 MiB memory in use. Including non-PyTorch memory, this process has 10.00 GiB memory in use. Of the allocated memory 9.78 GiB is allocated by PyTorch, and 52.28 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [17]:
import harreman

In [18]:
adata = harRunner.adata
adata

AnnData object with n_obs × n_vars = 112551 × 5006
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', '_scvi_batch', '_scvi_labels', 'leiden_scVI_res_0.5', 'leiden_scVI_res_2.5', 'leiden_scVI_res_2', 'leiden_scVI_res_1.5', 'leiden_scVI_res_1', 'leiden_scVI_res_0.75', 'leiden_scVI_res_0.65', 'leiden_scVI_res_0.375', 'leiden_scVI_res_0.25', 'leiden_scVI_res_0.1', 'leiden_scVI_res_0.05', 'cd8', 'cd4', 't_cell', 'Tier1', 'Tier2', 'Tier3', 'Cytotoxic_CD8_score', 'Exhausted_CD8_score', 'Treg_score', 'sub_cluster_5_res_1', 'sub_cluster_5_res_0.75', 'sub_cluster_5_res_0.5', 'sub_cluster_5_res_0.37', 'sub_cluster_5_res_0.25', 'sub_cluster_5_res_0.15', 'sub_cluster_5_res_0.1', 'sub_cluster_5_res_0.05', 'sub_cluster_2_res_1', 'sub_cluster_2_res_0.75', 'sub_cluster_2_res_0

In [19]:
a = adata.uns.keys()
a

dict_keys(['Tier1_colors', 'Tier2_colors', 'Tier3_colors', '_scvi_manager_uuid', '_scvi_uuid', 'cd4_colors', 'cd8_colors', 'leiden_scVI_res_0.05', 'leiden_scVI_res_0.05_colors', 'leiden_scVI_res_0.1', 'leiden_scVI_res_0.1_colors', 'leiden_scVI_res_0.25', 'leiden_scVI_res_0.25_colors', 'leiden_scVI_res_0.375', 'leiden_scVI_res_0.375_colors', 'leiden_scVI_res_0.5', 'leiden_scVI_res_0.5_colors', 'leiden_scVI_res_0.65', 'leiden_scVI_res_0.65_colors', 'leiden_scVI_res_0.75', 'leiden_scVI_res_0.75_colors', 'leiden_scVI_res_1', 'leiden_scVI_res_1.5', 'leiden_scVI_res_1.5_colors', 'leiden_scVI_res_1_colors', 'leiden_scVI_res_2', 'leiden_scVI_res_2.5', 'leiden_scVI_res_2.5_colors', 'leiden_scVI_res_2_colors', 'log1p', 'neighbors', 'rank_genes_groups', 'sub_cluster_2_res_0.05_colors', 'sub_cluster_2_res_0.15_colors', 'sub_cluster_2_res_0.1_colors', 'sub_cluster_2_res_0.25_colors', 'sub_cluster_2_res_0.37_colors', 'sub_cluster_2_res_0.5_colors', 'sub_cluster_2_res_0.75_colors', 'sub_cluster_2_res

In [20]:
# Memory-safe drop-in: accumulates the permutation null incrementally instead of
# allocating (n_cells, n_gene_pairs, M) arrays on the GPU (the stock version OOMs here,
# trying to grab ~114 GiB). Same outputs in adata.uns['interacting_cell_results'],
# minus the raw perm arrays. See metab_processing/interacting_cell_scores_lowmem.py
# and DataForClaude/documentation/05_harreman_reference.md (sec. 5).
from interacting_cell_scores_lowmem import compute_interacting_cell_scores_lowmem

compute_interacting_cell_scores_lowmem(adata, center_counts_for_np_test=False, test='both', restrict_significance='both', compute_significance='both', M=1000, seed=42, check_analytic_null=False, verbose=True)

# --- original (OOMs on large data) ---
# harreman.tools.compute_interacting_cell_scores(adata, center_counts_for_np_test=False, test='both', restrict_significance='both', compute_significance='both', M=1000, seed=42, check_analytic_null=False, verbose=True)


[lowmem] Computing gene pair and metabolite scores...
[lowmem] Running the parametric test...
[lowmem] Parametric test finished.
[lowmem] Running the non-parametric test...


[lowmem] Permutation test: 100%|██████████| 1000/1000 [01:33<00:00, 10.64it/s]


[lowmem] Non-parametric test finished.
[lowmem] Finished in 109.671 seconds


In [27]:
b = adata.uns.keys()
b

dict_keys(['Tier1_colors', 'Tier2_colors', 'Tier3_colors', '_scvi_manager_uuid', '_scvi_uuid', 'cd4_colors', 'cd8_colors', 'leiden_scVI_res_0.05', 'leiden_scVI_res_0.05_colors', 'leiden_scVI_res_0.1', 'leiden_scVI_res_0.1_colors', 'leiden_scVI_res_0.25', 'leiden_scVI_res_0.25_colors', 'leiden_scVI_res_0.375', 'leiden_scVI_res_0.375_colors', 'leiden_scVI_res_0.5', 'leiden_scVI_res_0.5_colors', 'leiden_scVI_res_0.65', 'leiden_scVI_res_0.65_colors', 'leiden_scVI_res_0.75', 'leiden_scVI_res_0.75_colors', 'leiden_scVI_res_1', 'leiden_scVI_res_1.5', 'leiden_scVI_res_1.5_colors', 'leiden_scVI_res_1_colors', 'leiden_scVI_res_2', 'leiden_scVI_res_2.5', 'leiden_scVI_res_2.5_colors', 'leiden_scVI_res_2_colors', 'log1p', 'neighbors', 'rank_genes_groups', 'sub_cluster_2_res_0.05_colors', 'sub_cluster_2_res_0.15_colors', 'sub_cluster_2_res_0.1_colors', 'sub_cluster_2_res_0.25_colors', 'sub_cluster_2_res_0.37_colors', 'sub_cluster_2_res_0.5_colors', 'sub_cluster_2_res_0.75_colors', 'sub_cluster_2_res

In [29]:
set(b) - set(a)

TypeError: unhashable type: 'AnnData'

In [41]:
adata.uns['interacting_cell_results']['np']['m']['pval']

array([[0.000999  , 0.01298701, 0.000999  , ..., 0.02297702, 0.03996004,
        0.05094905],
       [0.000999  , 0.00799201, 0.000999  , ..., 0.01998002, 0.03296703,
        0.05594406],
       [0.000999  , 0.00999001, 0.000999  , ..., 0.01698302, 0.05594406,
        0.05094905],
       ...,
       [0.000999  , 0.01198801, 0.000999  , ..., 0.01698302, 0.06093906,
        0.06793207],
       [0.000999  , 0.01098901, 0.000999  , ..., 0.01798202, 0.04195804,
        0.06293706],
       [0.000999  , 0.01098901, 0.000999  , ..., 0.02197802, 0.04395604,
        0.04995005]], shape=(112551, 141))